In [ ]:
from modelscope.pipelines import pipeline
from modelscope.utils.constant import Tasks
table_recognition = pipeline(Tasks.table_recognition, model='cv_dla34_table-structure-recognition_cycle-centernet模型的路徑')
result = table_recognition('你需要提取的圖片路徑')

In [ ]:
from paddleocr import PaddleOCR
ocr = PaddleOCR(use_gpu=True, lang='ch')
image_path = '你需要提取的圖片路徑'
res = ocr.ocr(image_path, cls=True)
print(res)

In [ ]:
from PIL import Image, ImageDraw, ImageFont
import textwrap
import numpy as np
def draw_ocr_boxes(image_path, boxes, texts):
   
    img = Image.open(image_path)
    img = Image.new('RGB', img.size, (255, 255, 255))
    
    draw = ImageDraw.Draw(img)
    font = ImageFont.truetype("./chinese_cht.ttf", size=15)  
    

    # 遍歷每個文字方塊和對應的文本
    for box, text in zip(boxes, texts):
        draw.rectangle(box, outline='red', width=2)
        x, y = box[:2]
        draw.text((x,y), text, font=font, fill='black')
    
    img.save('image_with_boxes_and_text.jpg')

# 示例文字方塊座標和對應的文字
boxes = [(*i[0][0],*i[0][2]) for i in res[0]]
texts = [i[1][0] for i in res[0]]
draw_ocr_boxes('你需要提取的圖片路徑', boxes, texts)

In [ ]:
def is_inside_text(cell, text):
    """檢查文字是否完全在儲存格內"""
    cx1, cy1, cx2, cy2 = cell
    tx1, ty1, tx2, ty2 = text['coords']
    return cx1 <= tx1 and cy1 <= ty1 and cx2 >= tx2 and cy2 >= ty2
def calculate_iou(cell, text):
    """
    計算兩個矩形框的交併比（IoU）。
    
    :param cell: 儲存格的座標 (x1, y1, x2, y2)
    :param text: 文字方塊的座標 (x1, y1, x2, y2)
    :return: 交併比（IoU）
    """
    # 計算交集的左上角和右下角座標
    intersection_x1 = max(cell[0], text['coords'][0])
    intersection_y1 = max(cell[1], text['coords'][1])
    intersection_x2 = min(cell[2], text['coords'][2])
    intersection_y2 = min(cell[3], text['coords'][3])

    # 如果沒有交集，回傳 0
    if intersection_x1 >= intersection_x2 or intersection_y1 >= intersection_y2:
        return 0.0

    # 計算交集的面積
    intersection_area = (intersection_x2 - intersection_x1) * (intersection_y2 - intersection_y1)

    # 計算並集的面積
    area_box1 = (cell[2] - cell[0]) * (cell[3] - cell[1])
    area_box2 = (text['coords'][2] - text['coords'][0]) * (text['coords'][3] - text['coords'][1])
    union_area = area_box1 + area_box2 - intersection_area

    # 計算 IoU
    iou = intersection_area / union_area

    return iou
def calculate_iot(cell, text):
    """
    計算兩個矩形框的交集面積和文字方塊面積的比值（IoT）。
    
    :param cell: 儲存格的座標 (x1, y1, x2, y2)
    :param text: 文字方塊的座標 (x1, y1, x2, y2)
    :return: IoT
    """
    # 計算交集的左上角和右下角座標
    intersection_x1 = max(cell[0], text['coords'][0])
    intersection_y1 = max(cell[1], text['coords'][1])
    intersection_x2 = min(cell[2], text['coords'][2])
    intersection_y2 = min(cell[3], text['coords'][3])

    # 如果沒有交集，回傳 0
    if intersection_x1 >= intersection_x2 or intersection_y1 >= intersection_y2:
        return 0.0
    # 計算交集的面積
    intersection_area = (intersection_x2 - intersection_x1) * (intersection_y2 - intersection_y1)

    text_area = (text['coords'][2] - text['coords'][0]) * (text['coords'][3] - text['coords'][1])
    # 計算 IoT
    iot = intersection_area / text_area
    return iot

def merge_text_into_cells(cell_coords, ocr_results):
    """將文字合併到儲存格"""
    # 建立一個字典，鍵是儲存格座標，值是屬於該儲存格的文字列表
    cell_text_dict = {cell: [] for cell in cell_coords}
    noncell_text_dict = {}
    
    # 遍歷 OCR 結果，將文字分配給正確的儲存格
    for cell in cell_coords:
        for result in ocr_results:
            if calculate_iot(cell, result)>0.5:
                cell_text_dict[cell].append(result['text'])
    
    for result in ocr_results:
        if all(calculate_iot(cell, result)<0.1 for cell in cell_coords):
            noncell_text_dict[result['coords']] = result['text']

    merged_text = {}
    for cell, texts in cell_text_dict.items():
        merged_text[cell] = ''.join(texts).strip()
    for coords, text in noncell_text_dict.items():
        merged_text[coords] = ''.join(text).strip()
    
    return merged_text

cell_coords = [tuple([*i[:2],*i[4:6]]) for i in result['polygons']]
ocr_results = [
    {'text': i[1][0], 'coords': tuple([*i[0][0],*i[0][2]])} for i in res[0]]
merged_text = merge_text_into_cells(cell_coords, ocr_results)
print(merged_text)

In [ ]:
from PIL import Image, ImageDraw, ImageFont
import textwrap
import numpy as np
def draw_text_boxes(image_path, boxes, texts):
    # 載入圖像
    img = Image.open(image_path)
    img = Image.new('RGB', img.size, (255, 255, 255))
    # 建立一個 ImageDraw 物件
    draw = ImageDraw.Draw(img)
    
    # 設定字型
    font = ImageFont.truetype("./chinese_cht.ttf", size=15)  # 選擇合適的字型和大小
    

    # 遍歷每個文字方塊和對應的文本
    for box, text in zip(boxes, texts):
        # 繪製文字方塊
        draw.rectangle(box, outline='red', width=2)
       
        
        text_len = draw.textbbox(xy=box[:2], text=text, font=font)
        
        if (text_len[2]-text_len[0]) > (box[2] - box[0]):
            # 如果文本長度大於文字方塊寬度,則將文本換行
            text = '\n'.join(textwrap.wrap(text, width=int(np.ceil((len(text) / np.ceil((text_len[2]-text_len[0]) / (box[2] - box[0])))))))
        else:
            # 否則直接繪製文本
            text = text
        x, y = box[:2]
        
        # 在文字方塊內居中文本
        draw.text((x,y), text, font=font, fill='black')
    
    # 儲存帶有文字方塊和文字的圖像
    img.save('你儲存的圖片路徑')

# 示例文字方塊座標和對應的文字
boxes = list(merged_text.keys())
texts = list(merged_text.values())
draw_text_boxes('你需要提取的圖片路徑', boxes, texts)

In [ ]:


def adjust_coordinates(merged_text, image_path):
    
    image = Image.open(image_path)
    width, height = image.size
    threshold = height / 100
    groups = {}
    
    for coordinates, text in merged_text.items():
        # 查找與當前 y 座標相差不超過 threshold 的分組
        found_group = False
        for group_y in groups.keys():
            if abs(coordinates[1] - group_y) <= threshold:
                groups[group_y].append((coordinates,text))
                found_group = True
                break

        # 如果沒有找到合適的分組，則建立一個新的分組
        if not found_group:
            groups[coordinates[1]] = [(coordinates,text)]
    
    # 計算每個分組的 y 座標的平均值，並更新座標列表
    adjusted_coordinates = {}
    for group_y, group_coords in groups.items():
        avg_y = sum(coord[0][1] for coord in group_coords) / len(group_coords)
        for i in group_coords:
            adjusted_coordinates[(i[0][0], avg_y, i[0][2], i[0][3])] = i[1]
        

    return adjusted_coordinates

# 呼叫函數處理座標
adjusted_merged_text = adjust_coordinates(merged_text, '你需要提取的圖片路徑')

# 印出結果
print("原始座標:", merged_text)
print("調整後的座標:", adjusted_merged_text)

In [ ]:
from PIL import Image, ImageDraw, ImageFont
import textwrap
import numpy as np
def draw_text_boxes(image_path, boxes, texts):
   
    img = Image.open(image_path)
    img = Image.new('RGB', img.size, (255, 255, 255))
    draw = ImageDraw.Draw(img)
    font = ImageFont.truetype("./chinese_cht.ttf", size=15)  # 選擇合適的字型和大小
    for box, text in zip(boxes, texts):
        
        draw.rectangle(box, outline='red', width=2)
       
        
        text_len = draw.textbbox(xy=box[:2], text=text, font=font)
        
        if (text_len[2]-text_len[0]) > (box[2] - box[0]):
            # 如果文本長度大於文字方塊寬度,則將文本換行
            text = '\n'.join(textwrap.wrap(text, width=int(np.ceil(len(text) / np.ceil((text_len[2]-text_len[0]) / (box[2] - box[0]))))))
        else:
            # 否則直接繪製文本
            text = text
        x, y = box[:2]
        
        draw.text((x,y), text, font=font, fill='black')
    img.save('你需要儲存的圖片路徑')

boxes = list(adjusted_merged_text.keys())
texts = list(adjusted_merged_text.values())
draw_text_boxes('你需要提取的圖片路徑', boxes, texts)

In [ ]:
#輸出最終的文本
adjusted_merged_text_sorted = sorted(adjusted_merged_text.items(), key=lambda x: (x[0][1], x[0][0]))
adjusted_merged_text_sorted_group = {}
for coordinates, text in adjusted_merged_text_sorted:
    if coordinates[1] not in adjusted_merged_text_sorted_group:
        adjusted_merged_text_sorted_group[coordinates[1]] = [text]
    else:
        adjusted_merged_text_sorted_group[coordinates[1]].append(text)
for text_list in adjusted_merged_text_sorted_group.values():
    print(' | '.join(text_list))
